In [0]:
from pyspark.sql import functions as F

acs = spark.read.csv("/Volumes/workspace/default/health_data_cs483/ACS_2016_2020.csv",
                     header=True, inferSchema=True)

acs_clean = (acs
    .drop("OBJECTID_1", "Shape__Area", "Shape__Length")   # map/ID columns, not needed
    .filter(F.col("NAME") != "Puerto Rico")                # other datasets are states + DC
    .withColumnRenamed("GEOID", "statefips")               # join key for later
    .withColumnRenamed("STUSAB", "state_abbr")
    .withColumnRenamed("NAME", "state_name")
    .withColumnRenamed("B19013EST1", "median_household_income")
    .withColumnRenamed("B17021EST2_PCT", "poverty_pct")
    .withColumnRenamed("B23001_UE_PCT", "unemployment_pct"))

print(acs.count(), "→", acs_clean.count())
acs_clean.write.mode("overwrite").saveAsTable("workspace.default.acs_clean")

display(spark.table("workspace.default.acs_clean"))

In [0]:
# Checks
from pyspark.sql.types import StringType
import matplotlib.pyplot as plt
import numpy as np
import os

nulls = acs.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in acs.columns]).first().asDict()
print("Columns with missing values:", {c: n for c, n in nulls.items() if n > 0} or "none")

print("Duplicate rows:", acs.count() - acs.dropDuplicates().count())
print("Duplicate GEOIDs:", acs.count() - acs.select("GEOID").distinct().count())

text_cols = [f.name for f in acs.schema.fields if isinstance(f.dataType, StringType)]
print("Text columns:", text_cols)

print("Names with extra spaces:", acs.filter(F.col("NAME") != F.trim("NAME")).count())
print("State codes not 2 letters:", acs.filter(F.length("STUSAB") != 2).count())
pct_cols = [c for c in acs.columns if c.endswith("_PCT")]
bad = acs.select([F.sum(((F.col(c) < 0) | (F.col(c) > 100)).cast("int")).alias(c) for c in pct_cols]).first()
print("Percent values outside 0-100:", sum(bad))

In [0]:
# Histograms
pdf = acs_sel.toPandas()

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, c in zip(axes.flat, feat):
    ax.hist(pdf[c], bins=12, color="#4C72B0", edgecolor="white")
    ax.axvline(pdf[c].median(), color="#DD8452", linestyle="--", label="median")
    ax.set_title(c); ax.set_xlabel("value"); ax.set_ylabel("states")
axes.flat[0].legend()
fig.suptitle("Distribution of ACS features (50 states + DC)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "acs_histograms.png"), dpi=110)
plt.show()

In [0]:
MORE_FEATURES = {
    "B19001_GT150_PCT": "income_150k_plus_pct",
    "B08013_AVG_TTW": "avg_commute_min",
    "B08303_60PLUS_TTW_PCT": "commute_60plus_pct",
    "B25014_CROWD_PCT": "crowded_housing_pct",
    "B25106_CB_PCT": "housing_cost_burden_pct",
    "C24010_SERVICE_PCT": "service_jobs_pct",
    "C24010_MPRO_PCT": "mgmt_prof_jobs_pct",
}
for old, new in MORE_FEATURES.items():
    acs_clean = acs_clean.withColumnRenamed(old, new)

feat = ["median_household_income", "poverty_pct", "unemployment_pct"] + list(MORE_FEATURES.values())
acs_sel = acs_clean.select("statefips", "state_abbr", "state_name", *feat)
display(acs_sel.limit(5))

In [0]:
# Summary statistics
display(acs_sel.select(feat).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max"))
display(acs_sel.select([F.round(F.skewness(c), 2).alias(c) for c in feat]))
print("Unique states:", acs_sel.select("state_abbr").distinct().count())

In [0]:
# Folder for plots
REPO = os.path.dirname(os.getcwd())
IMG_DIR = os.path.join(REPO, "doc", "eda")
os.makedirs(IMG_DIR, exist_ok=True)
print("Saving to:", IMG_DIR)

In [0]:
#Histograms
pdf = acs_sel.toPandas()

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, c in zip(axes.flat, feat):
    ax.hist(pdf[c], bins=12, color="#4C72B0", edgecolor="white")
    ax.axvline(pdf[c].median(), color="#DD8452", linestyle="--", label="median")
    ax.set_title(c); ax.set_xlabel("value"); ax.set_ylabel("states")
axes.flat[0].legend()
fig.suptitle("Distribution of ACS features (50 states + DC)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "acs_histograms.png"), dpi=110)
plt.show()

In [0]:
#Box plots
z = (pdf[feat] - pdf[feat].mean()) / pdf[feat].std()
fig, ax = plt.subplots(figsize=(14, 6))
ax.boxplot([z[c] for c in feat], vert=False, patch_artist=True,
           boxprops=dict(facecolor="#A1C9F4"), flierprops=dict(markerfacecolor="#C44E52"))
ax.set_yticks(range(1, len(feat) + 1)); ax.set_yticklabels(feat)
ax.set_xlabel("z-score"); ax.set_title("Standardized ACS features (red = outliers)")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "acs_boxplots.png"), dpi=110)
plt.show()

In [0]:
#Correlation heatmap
pairs = [(a, b) for i, a in enumerate(feat) for b in feat[i + 1:]]
r = acs_sel.select([F.corr(a, b).alias(f"{a}|{b}") for a, b in pairs]).first().asDict()
corr = np.eye(len(feat))
for a, b in pairs:
    i, j = feat.index(a), feat.index(b)
    corr[i, j] = corr[j, i] = r[f"{a}|{b}"]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feat))); ax.set_xticklabels(feat, rotation=45, ha="right")
ax.set_yticks(range(len(feat))); ax.set_yticklabels(feat)
for i in range(len(feat)):
    for j in range(len(feat)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Pearson r"); ax.set_title("Correlation between ACS features")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "acs_correlation.png"), dpi=110)
plt.show()

In [0]:
# Scatter plot
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(pdf["median_household_income"], pdf["poverty_pct"], color="#55A868")
for _, row in pdf.iterrows():
    ax.annotate(row["state_abbr"], (row["median_household_income"], row["poverty_pct"]), fontsize=8)
ax.set_xlabel("Median household income ($)"); ax.set_ylabel("Poverty (%)")
ax.set_title("Income vs. poverty by state")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "acs_income_vs_poverty.png"), dpi=110)
plt.show()

In [0]:
# Outliers
rows = []
for c in feat:
    q1, q3 = acs_sel.approxQuantile(c, [0.25, 0.75], 0.0)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mean, std = acs_sel.select(F.mean(c), F.stddev(c)).first()
    flagged = (acs_sel.withColumn("z", (F.col(c) - mean) / std)
               .filter((F.col(c) < lo) | (F.col(c) > hi) | (F.abs("z") > 3))
               .select("state_abbr", c, F.round("z", 2).alias("z")).collect())
    for f in flagged:
        rows.append((c, f["state_abbr"], float(f[c]), float(f["z"]),
                     bool(f[c] < lo or f[c] > hi), bool(abs(f["z"]) > 3)))

outliers = spark.createDataFrame(rows, "feature string, state string, value double, z double, iqr_outlier boolean, z_gt_3 boolean")
display(outliers.orderBy("feature", F.desc("z")))

### Outlier decision
Outliers: NY, NJ, MD (long commutes), HI, CA, AK (crowded housing), DC (professional jobs), NV and DC (service jobs).
**Kept all of them.** They are real states, not errors. With 51 rows, removing them would lose real variation. Scale differences are handled with z-scores.

In [0]:
# Scaling
display(acs_sel.select([(F.max(c) - F.min(c)).alias(c) for c in feat]))

stats = acs_sel.select([F.mean(c).alias(c + "_m") for c in feat] +
                       [F.stddev(c).alias(c + "_s") for c in feat]).first()
acs_scaled = acs_sel.select("statefips", "state_abbr", "state_name",
    *[((F.col(c) - stats[c + "_m"]) / stats[c + "_s"]).alias(c + "_z") for c in feat])
display(acs_scaled.limit(5))

In [0]:
# Tables that will be used for building different models
acs_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.acs_clean")
acs_sel.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.acs_selected")
acs_scaled.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.acs_scaled")
print("Saved 3 tables")

### Key takeaways
- No missing values, duplicates, bad formatting, or invalid percents. Only Puerto Rico and map columns removed.
- Income, % earning $150K+, and % professional jobs are highly correlated (r = 0.80–0.96).
- Income and poverty are strongly negatively related (r = -0.72).
- Unemployment tracks poverty (r = 0.58) but not income.
- Average commute and % 60+ min commutes are nearly the same signal (r = 0.91).
- Features must be standardized before SVD, K-means, and regression.